In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Spam Classification with DistilBERT (PyTorch + HuggingFace)

Created a custom Trainer and injected the cost senstive in it. https://huggingface.co/transformers/v4.0.1/main_classes/trainer.html

## Start

In [2]:
import os
import joblib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy

# Splitting the model into train-test-validation that be 70-20-10 repectively
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Metrics
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report)

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [3]:
# Abstract Method:
# 1. Baseline - Done
# 2. Cleaned - Done
# 3. NER - Done

# Datasets Type:
# Enron - Done
# SpamAssassin - Done
# LingSpam - Done

# Model Type:
# DistilBERT - Done
# BERT - Done
abstract = "Cleaned"
dataset = "SpamAssassin"
model_name = "BERT"

In [4]:
df = pd.read_csv(f'drive/MyDrive/Dissertation/Datasets/{abstract}/{dataset}/{dataset}_{abstract}.csv')

In [5]:
# Fix the print
before = len(df)
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
after = len(df)
print(f"Before dedup: {before}")
print(f"After dedup: {after}")
print(f"Removed: {before - after} rows")

Before dedup: 6047
After dedup: 6047
Removed: 0 rows


In [6]:
# First split
X_train80, X_test, y_train80, y_test = train_test_split(
    df['text'], # processed_text
    df['label'],
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

# Validation split
val_percent = 0.10 / (1 - 0.2)

X_train70, X_val, y_train70, y_val = train_test_split(
    X_train80,
    y_train80,
    test_size=val_percent,
    stratify=y_train80,
    random_state=42
)

In [7]:
train_ds = Dataset.from_dict({
    "text": X_train70.tolist(), # processed_text
    "label": y_train70.astype(int).tolist()
})

val_ds = Dataset.from_dict({
    "text": X_val.tolist(),
    "label": y_val.astype(int).tolist()
})

test_ds = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": y_test.astype(int).tolist()
})

In [8]:
train_texts = set(train_ds['text'])
val_texts = set(train_ds['text'])
overlap = train_texts.intersection(val_texts)
print(f"Overlapping samples: {len(overlap)}")

Overlapping samples: 4232


In [9]:
from collections import Counter
print("Train distribution:", Counter(train_ds["label"]))
print("Val distribution:", Counter(val_ds["label"]))
print("Test distribution:", Counter(test_ds["label"]))

Train distribution: Counter({0: 2905, 1: 1327})
Val distribution: Counter({0: 415, 1: 190})
Test distribution: Counter({0: 830, 1: 380})


In [10]:
tokenizer = AutoTokenizer.from_pretrained(f'{model_name}-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained(f'{model_name}-base-uncased', num_labels=1)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: BERT-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# Definig custom tokens
custom_ner_tokens : list[str] = [
    "EMAILNER",
    "PHONENER",
    "WEBNER",
    "DATENER",
    "TIMENER",
    "MONEYNER",
    "PERCENTNER",
    "PERNER",
    "ORGNER",
    "GPENER",
    "LOCNER",
    "PRODUCTNER",
    "EVENTNER",
    "NORPNER",
    "LANGNER",
    "NUMNER",
]

# Register to the tokenizer treating them as a single unit
tokenizer.add_special_tokens({"additional_special_tokens": custom_ner_tokens})

16

In [12]:
model.resize_token_embeddings(len(tokenizer))

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(30538, 768, padding_idx=0)

In [13]:
def tokenize(text):
    return tokenizer(text["text"], truncation=True, max_length=512) # processed_text

In [14]:
train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/4232 [00:00<?, ? examples/s]

Map:   0%|          | 0/605 [00:00<?, ? examples/s]

Map:   0%|          | 0/1210 [00:00<?, ? examples/s]

In [15]:
print(test_ds)

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1210
})


In [16]:
dynamic_padding = DataCollatorWithPadding(tokenizer=tokenizer)

## Computing Class Weight

In [17]:
classes = np.unique(train_ds["label"])

cw = compute_class_weight(
    'balanced',
    classes=classes,
    y=train_ds["label"]
)

class_weights = {
    int(k): float(v)
    for k, v in zip(classes, cw)
}

print(class_weights)

{0: 0.7283993115318417, 1: 1.5945742275810098}


In [18]:
def compute_metrics(pred):
    # 1. Extract logits and labels
    logits = pred.predictions.squeeze(-1)  # [batch, 1] -> [batch,]
    labels = pred.label_ids  # Extract labels from pred object

    # 2. Convert logits to probabilities
    probs = 1 / (1 + np.exp(-logits))  # Sigmoid

    # 3. Find OPTIMAL threshold on validation data using PR curve
    precision, recall, thresholds = precision_recall_curve(labels, probs)
    f1_scores = 2 * recall * precision / (recall + precision + 1e-10)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]

    # 4. Apply optimal threshold for predictions
    preds = (probs >= best_threshold).astype(int)

    # 5. Compute metrics
    precision_final, recall_final, f1_final, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    pr_auc = average_precision_score(labels, probs)  # Use probs, not preds

    return {
        'accuracy': acc,
        'precision': precision_final,
        'recall': recall_final,
        'f1': f1_final,
        'pr_auc': pr_auc,
        'best_threshold': float(best_threshold)  # Report threshold used
    }

## Visualization of Results

In [19]:
class MetricsVisualizationCallback(TrainerCallback):

  """
  Custom Hugging Face Trainer callback to log training metrics and generate
  plots.

  This callback leverages:
    - on_log: To obtain the model's training loss format during iterations.
    - on_evaluate: To calculate comprehensive threshold-based metrics for
    validation datasets.
    - on_train_end: To build, format, and export final historical plots.


  Attributes:
    output_dir: Represents the path to the output directory
    model_name: Represents the DistilBert/Bertcase that is used during the trainig
    train_dataset: Represents the input dataset that used for model training
    val_dataset: Represents the validation dataset used for tuning and evaluation

  Returns:
    Saves structured metrics in CSV format and tracking plots in the designated
    output folders
  """

  def __init__(self, output_dir, model_name, train_dataset, val_dataset):
    self.output_dir   = output_dir
    self.model_name   = model_name

    self.val_dataset        = val_dataset
    self.train_dataset      = train_dataset
    self.latest_train_loss  = None

    # Saving the metrics
    self.metrics_history  = []

  def on_log(self, args, state, control, logs=None, **kwargs):
    if logs is None:
      return

    if "loss" in logs:
      self.latest_train_loss = float(logs["loss"])

  def on_evaluate(self, args, state, control, metrics=None, **kwargs):

    """Store metrics only, no plotting"""
    if metrics is None:
        return control

    epoch = int(state.epoch)

    """
    +++++++++++++++++++++
    +   Train Dataset   +
    +++++++++++++++++++++
    """;

    train_predictions = self.trainer.predict(self.train_dataset)
    train_logits      = train_predictions.predictions.squeeze(-1)
    train_labels      = train_predictions.label_ids
    train_probs       = 1 / (1 + np.exp(-train_logits))

    train_precision_curve, train_recall_curve, train_pr_thresholds = precision_recall_curve(train_labels, train_probs)
    train_pr_auc = average_precision_score(train_labels, train_probs)
    train_f1_scores = 2 * train_recall_curve * train_precision_curve / (train_recall_curve + train_precision_curve + 1e-10)

    train_best_idx       = np.argmax(train_f1_scores[:-1])
    train_best_threshold = train_pr_thresholds[train_best_idx]

    train_preds = (train_probs >= train_best_threshold).astype(int)

    train_accuracy = accuracy_score(train_labels, train_preds)

    """
    +++++++++++++++++++
    +   Val Dataset   +
    +++++++++++++++++++
    """;

    val_predictions = self.trainer.predict(self.val_dataset)
    val_logits      = val_predictions.predictions.squeeze(-1)
    val_labels      = val_predictions.label_ids
    val_probs       = 1 / (1 + np.exp(-val_logits))

    val_preds = (val_probs >= train_best_threshold).astype(int)

    val_accuracy = accuracy_score(val_labels, val_preds)

    """
    +++++++++++++++++++++++
    +   Metrics Dataset   +
    +++++++++++++++++++++++
    """;

    cm = confusion_matrix(val_labels, val_preds)

    tn, fp, fn, tp = cm.ravel()

    fpr, tpr, _ = roc_curve(val_labels, val_probs)
    roc_auc     = auc(fpr, tpr)
    pr_auc      = average_precision_score(val_labels, val_probs)

    """
    ++++++++++++++++++++++
    +   Saving Metrics   +
    ++++++++++++++++++++++
    """;

    epoch_metrics = {
        'epoch' : int(epoch),

        # Saving the loss values
        "train_loss" : self.latest_train_loss,
        'eval_loss'  : metrics["eval_loss"],

        # Saving the accuracy values
        'train_accuracy' : float(train_accuracy),
        'eval_accuracy'  : float(val_accuracy),

        # Other scores
        'precision' : float(precision_score(val_labels, val_preds)),
        'recall'    : float(recall_score(val_labels, val_preds)),
        'f1'        : float(f1_score(val_labels, val_preds)),

        'pr_auc'         : float(pr_auc),
        'train_pr_auc' : float(train_pr_auc),
        'roc_auc'        : float(roc_auc),
        'best_threshold' : float(train_best_threshold),

        # Confusion Matrix
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)
    }
    self.metrics_history.append(epoch_metrics)

    # Saving to CSV
    csv_path = f"{self.output_dir}/Results/{self.model_name}_metrics.csv"
    pd.DataFrame(self.metrics_history).to_csv(csv_path,index=False)

    return control

  def on_train_end(self, args, state, control, **kwargs):

    """Generate all final plots and summary"""

    if not hasattr(self, 'trainer') or self.trainer is None or len(self.metrics_history) == 0:
        print("No metrics to plot.")
        return control

    # Reusing the matrix from on_evaluate method.
    best_metrics = self.metrics_history[-1]

    # Plotting Confusion Matrix require to call the tp, fp, fn, tn, and
    # join them using np.array()
    confusion_matrix_plot = np.array(
        [[best_metrics["tn"], best_metrics["fp"]],
          [best_metrics["fn"], best_metrics["tp"]]])

    # Plotting Confusion Matrix
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix_plot, display_labels=['Ham', 'Spam'])
    disp.plot(cmap='Blues', ax=ax)
    plt.title(f'{self.model_name} Confusion Matrix')
    plt.savefig(f"{self.output_dir}/Image/Bert/{self.model_name}_confusion_matrix.png", dpi=300, bbox_inches='tight')
    plt.close()

    # Reconstructing the ROC-AUC
    predictions = self.trainer.predict(self.val_dataset)
    logits = predictions.predictions.squeeze(-1)
    labels = predictions.label_ids
    probs = 1 / (1 + np.exp(-logits))

    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)

    # Plotting ROC Curve
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], linestyle='--')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{self.model_name} ROC Curve')
    ax.legend(loc='lower right')
    ax.grid(True)
    plt.savefig(f"{self.output_dir}/Image/Bert/{self.model_name}_roc_curve.png", dpi=300, bbox_inches='tight')
    plt.close()

    # Recontructing the PR-AUC
    precision_curve, recall_curve, pr_thresholds = precision_recall_curve(labels, probs)
    pr_auc = average_precision_score(labels, probs)

    # Plotting PR Curve
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(recall_curve, precision_curve, lw=2, label=f'PR (AP = {pr_auc:.3f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'{self.model_name} Precision-Recall Curve')
    ax.legend(loc='lower left')
    ax.grid(True)
    plt.savefig(f"{self.output_dir}/Image/Bert/{self.model_name}_pr_curve.png", dpi=300, bbox_inches='tight')
    plt.close()

    best_epoch = max(
      self.metrics_history,
      key=lambda x: x['f1']
    )

    best_threshold = best_epoch['best_threshold']
    best_f1 = best_epoch['f1']
    f1_scores = (
        2 * precision_curve * recall_curve /
        (precision_curve + recall_curve + 1e-10)
    )

    # Plotting Threshold vs F1 Score
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(pr_thresholds, f1_scores[:-1], lw=2, label=f'F1 Score')

    plt.axvline(best_threshold, linestyle="--", label=f"Best Threshold = {best_threshold:.3f}")
    plt.scatter([best_threshold], [best_f1], s=80,label=f"Best F1 = {best_f1:.3f}")

    ax.set_xlabel('Threshold')
    ax.set_ylabel('F1 Score')
    ax.set_title(f'{self.model_name} Threshold vs F1 Score')
    ax.legend(loc='lower left')
    ax.grid(True)
    plt.savefig(f"{self.output_dir}/Image/Bert/{self.model_name}_threshold_vs_f1.png", dpi=300, bbox_inches='tight')
    plt.close()

    # PR-AUC
    pr_auc_values       = [m['pr_auc'] for m in self.metrics_history]
    train_pr_auc_values = [m['train_pr_auc'] for m in self.metrics_history]
    epochs              = [m['epoch'] for m in self.metrics_history]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(epochs, train_pr_auc_values, label='train_pr_auc', color='blue')
    ax.plot(epochs, pr_auc_values,       label='val_pr_auc',   color='orange')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('PR-AUC')
    ax.set_title(f'{self.model_name} PR-AUC per Epoch')
    ax.legend()
    ax.grid(True)
    plt.savefig(f"{self.output_dir}/Image/Bert/{self.model_name}_pr_auc_per_epoch.png", dpi=300, bbox_inches='tight')
    plt.close()

    # Training History
    self._plot_training_history(state)

    # Print Final Summary
    print(f"\n{'='*50}")
    print(f"\nAll plots saved to: {self.output_dir}/Image/Bert")

    return control

  def _plot_training_history(self, state):
    """Plot train/val loss from metrics history"""
    logs_df = pd.DataFrame(self.metrics_history)

    fig, axes = plt.subplots(2, 1, figsize=(10, 20))

    # Train and Test Accuracy
    train_epoch = logs_df.groupby("epoch", as_index=False)["train_accuracy"].mean()
    eval_acc = logs_df[logs_df['eval_accuracy'].notna()]

    axes[0].plot(train_epoch['epoch'], train_epoch['train_accuracy'], label='Train')
    axes[0].plot(eval_acc['epoch'], eval_acc['eval_accuracy'], label='Val')
    axes[0].set_title(f'{self.model_name} Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('accuracy')
    axes[0].legend()
    axes[0].grid(True)

    # Train and Test Loss
    train_epoch = logs_df.groupby("epoch", as_index=False)["train_loss"].mean()
    eval_loss = logs_df[logs_df['eval_loss'].notna()]

    axes[1].plot(train_epoch['epoch'], train_epoch['train_loss'], label='Train')
    axes[1].plot(eval_loss['epoch'], eval_loss['eval_loss'], label='Val')
    axes[1].set_title(f'{self.model_name} Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(f"{self.output_dir}/Image/Bert/{self.model_name}_training_history.png", dpi=300, bbox_inches='tight')
    plt.close()

In [20]:
# Initiate the callback
metrics_callback = MetricsVisualizationCallback(
    output_dir    = f'drive/MyDrive/Dissertation/Results/{abstract}/{dataset}',
    model_name    = f"{model_name}",
    train_dataset = train_ds,
    val_dataset   = val_ds
)

## Training and Testing Data

In [21]:
class WeightComputing(Trainer):
  """
  Creating a compute_loss method to inject the class weights to the Trainer loss function

  Attributes:
    model: Represents the DistilBert/Bertcase that is used during the trainig
    input (dict): Represents a dictionary containing input_ids, attention_mask, and labels
    return_outputs (bool): Determinig if the model returns the loss or (loss and logits)

  Returns:
    Calculated loss value.
  """
  def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
    # Extracting the labels from the input and apply the forward pass
    labels = inputs.pop("labels").float() # Label Extraction
    labels = labels.unsqueeze(1) # Matching logits shapes

    outputs = model(**inputs) # Forward Pass
    logits  = outputs.get("logits") # Shape: [batch_size, 1]

    # Initialize the BCE loss function with pre-computed class weights
    pos_weight = torch.tensor(class_weights[1]/class_weights[0], dtype=torch.float).to(logits.device) # Computing the positive weight
    loss_class_weight = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # Calculate the cross-entropy loss
    loss = loss_class_weight(logits, labels)

    return (loss, outputs) if return_outputs else loss

In [22]:
training_args = TrainingArguments(
    output_dir=f'drive/MyDrive/Dissertation/Results/{abstract}/{dataset}/Features/Checkpoint',
    save_total_limit=1,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=10,

    # Batch size = 8 * 2 = 16
    # Train on the first 8 emails
    # Train on the second 8 emails
    # Compute the gradient
    # Update the weight
    # Back to step 1.
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,  # keeps effective batch size at 16

    num_train_epochs=50,
    learning_rate=2e-5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model='pr_auc',
    bf16=True, # versatile than fp16
    report_to='none', # If set to True, your training will be set to cloud Azure storage and others, not your bucket.
    seed=42,
)

In [23]:
trainer = WeightComputing(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=dynamic_padding,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),
        metrics_callback]
)


trainer.save_model(f'drive/MyDrive/Dissertation/Results/{abstract}/{dataset}/Predicted/{model_name}_Trainer')
tokenizer.save_pretrained(f'drive/MyDrive/Dissertation/Results/{abstract}/{dataset}/Features/{model_name}_Tokenizer')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('drive/MyDrive/Dissertation/Results/Cleaned/SpamAssassin/Features/BERT_Tokenizer/tokenizer_config.json',
 'drive/MyDrive/Dissertation/Results/Cleaned/SpamAssassin/Features/BERT_Tokenizer/tokenizer.json')

In [24]:
# Link callback to trainer (needed for predictions)
metrics_callback.trainer = trainer

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Pr Auc,Best Threshold
1,0.959524,0.222752,0.975207,0.983425,0.936842,0.959569,0.978558,0.126785
2,0.296817,0.152266,0.985124,0.978836,0.973684,0.976253,0.990656,0.066569
3,0.614039,0.334187,0.981818,0.989071,0.952632,0.970509,0.994730,0.006098
4,0.012764,0.141833,0.986777,0.978947,0.978947,0.978947,0.993332,0.026355
5,0.209630,0.167644,0.986777,0.989247,0.968421,0.978723,0.993624,0.054199
6,0.001882,0.231952,0.983471,0.989130,0.957895,0.973262,0.990855,0.175538


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte



All plots saved to: drive/MyDrive/Dissertation/Results/Cleaned/SpamAssassin/Image/Bert


TrainOutput(global_step=1590, training_loss=0.357974789144893, metrics={'train_runtime': 518.9411, 'train_samples_per_second': 407.753, 'train_steps_per_second': 25.533, 'total_flos': 6680855932452864.0, 'train_loss': 0.357974789144893, 'epoch': 6.0})

In [25]:
# test_results = trainer.evaluate(eval_dataset=test_ds)

# last = metrics_callback.metrics_history[-1]

# pd_results = pd.DataFrame([test_results]).rename(columns={
#     'eval_accuracy'       : 'Accuracy',
#     'eval_precision'      : 'Precision',
#     'eval_recall'         : 'Recall',
#     'eval_f1'             : 'F1-Score',
#     'eval_best_threshold' : 'Threshold',
#     'eval_pr_auc'         : 'PR-AUC',
# }).assign(
#     AUC=float(last['roc_auc']),
#     TP=int(last['tp']),
#     TN=int(last['tn']),
#     FP=int(last['fp']),
#     FN=int(last['fn']),
# )[[
#     'Accuracy', 'Precision', 'Recall',
#     'F1-Score', 'Threshold', 'AUC', 'PR-AUC', 'TP', 'TN', 'FP', 'FN'
# ]]

# --- Get the validation-derived threshold (from training-time tracking) ---
last = metrics_callback.metrics_history[-1]
final_val_threshold = last['best_threshold']

# --- Predict on TEST — bypass compute_metrics' internal threshold search ---
test_predictions = trainer.predict(test_ds)
test_logits = test_predictions.predictions.squeeze(-1)
test_labels = test_predictions.label_ids
test_probs = 1 / (1 + np.exp(-test_logits))   # sigmoid

# --- Apply the FIXED validation threshold — no new threshold search on test ---
test_preds = (test_probs >= final_val_threshold).astype(int)

test_accuracy = accuracy_score(test_labels, test_preds)
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average='binary'
)
test_pr_auc = average_precision_score(test_labels, test_probs)

# --- Confusion matrix on TEST (not reused from validation's `last`) ---
cm = confusion_matrix(test_labels, test_preds)
tn, fp, fn, tp = cm.ravel()

fpr, tpr, _ = roc_curve(test_labels, test_probs)
test_auc = auc(fpr, tpr)

pd_results = pd.DataFrame([{
    'Accuracy'  : test_accuracy,
    'Precision' : test_precision,
    'Recall'    : test_recall,
    'F1-Score'  : test_f1,
    'Threshold' : final_val_threshold,
    'AUC'       : test_auc,
    'PR-AUC'    : test_pr_auc,
    'TP'        : int(tp),
    'TN'        : int(tn),
    'FP'        : int(fp),
    'FN'        : int(fn),
}])[[
    'Accuracy', 'Precision', 'Recall',
    'F1-Score', 'Threshold', 'AUC', 'PR-AUC', 'TP', 'TN', 'FP', 'FN'
]]

pd_results.to_csv(f'drive/MyDrive/Dissertation/Results/{abstract}/{dataset}/Results/{model_name}_test_results.csv', index=False)

In [26]:
pd_results

,Accuracy,Precision,Recall,F1-Score,Threshold,AUC,PR-AUC,TP,TN,FP,FN
0,0.966116,1.0,0.892105,0.942976,0.66628,0.997387,0.995301,339,830,0,41
